# AnalystLab Africa ML Internship — Week 4
## Supervised Learning: Regression & Classification
**Task 1:** Linear Regression — House Price Prediction  
**Task 2:** Logistic Regression — Titanic Survival Classification  
**Author:** ML Intern  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (mean_squared_error, r2_score,
                             accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print("✅ Libraries loaded successfully.")

---
# TASK 1 — Linear Regression: House Price Prediction

## Problem Statement
Predict the **median house value** (`medv`) of Boston neighbourhoods based on features such as number of rooms, crime rate, pupil-teacher ratio, and lower-status population percentage.

**Dataset:** Boston Housing Dataset (506 samples, 13 features)  
**Target:** `medv` — Median value of owner-occupied homes in $1000s  
**Algorithm:** Linear Regression

## Step 2 — Load Dataset

In [ ]:
house = pd.read_csv('house_prices.csv')
print(f"Shape: {house.shape}")
print("\nColumn descriptions:")
col_desc = {
    'crim':    'Per capita crime rate by town',
    'zn':      'Proportion of residential land zoned for large lots',
    'indus':   'Proportion of non-retail business acres per town',
    'chas':    'Charles River dummy variable (1 if tract bounds river)',
    'nox':     'Nitric oxides concentration',
    'rm':      'Average number of rooms per dwelling',
    'age':     'Proportion of owner-occupied units built before 1940',
    'dis':     'Weighted distances to employment centres',
    'rad':     'Index of accessibility to radial highways',
    'tax':     'Full-value property-tax rate per $10,000',
    'ptratio': 'Pupil-teacher ratio by town',
    'b':       'Proportion of Black residents by town',
    'lstat':   'Percentage of lower-status population',
    'medv':    'TARGET: Median home value in $1000s'
}
for col, desc in col_desc.items():
    print(f"  {col:10} — {desc}")

## Step 3 — Explore the Dataset

In [ ]:
print("First 5 rows:")
house.head()

In [ ]:
print("Data types and missing values:")
print(house.dtypes)
print("\nMissing values:", house.isnull().sum().sum(), "— No missing data ✅")
print("\nStatistical summary:")
house.describe().round(2)

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(house['medv'], bins=30, color='#2563eb', edgecolor='white')
axes[0].set_title('Distribution of House Prices (medv)')
axes[0].set_xlabel('Median Value ($1000s)')
axes[0].set_ylabel('Count')

# Correlation heatmap
corr = house.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', ax=axes[1],
            annot=False, linewidths=0.3)
axes[1].set_title('Feature Correlation Heatmap')
plt.suptitle('House Prices — EDA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('house_eda.png', bbox_inches='tight')
plt.show()

In [ ]:
# Key relationships with target
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feat, color in zip(axes, ['rm','lstat','ptratio'],
                            ['#2563eb','#ef4444','#22c55e']):
    ax.scatter(house[feat], house['medv'], alpha=0.4, color=color, s=15)
    ax.set_xlabel(feat)
    ax.set_ylabel('medv (House Price)')
    ax.set_title(f'{feat} vs Price')
plt.suptitle('Feature Relationships with House Price', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('house_scatter.png', bbox_inches='tight')
plt.show()
print("rm (rooms) shows a strong positive correlation with price.")
print("lstat (lower-status %) shows a strong negative correlation.")

## Step 4 — Select Features and Target

In [ ]:
# Use key features most correlated with price
features = ['rm', 'lstat', 'ptratio', 'crim', 'nox', 'dis', 'tax', 'age']
X = house[features]
y = house['medv']

print(f"Features selected: {features}")
print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget stats:")
print(f"  Min price:  ${y.min():.1f}k")
print(f"  Max price:  ${y.max():.1f}k")
print(f"  Mean price: ${y.mean():.1f}k")

## Step 5 — Split the Dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training samples: {len(X_train)} (80%)")
print(f"Testing samples:  {len(X_test)} (20%)")

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("\n✅ Features scaled using StandardScaler.")

## Step 6 — Train Linear Regression Model

In [ ]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
print("✅ Linear Regression model trained.")
print("\nModel coefficients:")
coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': lr.coef_.round(3)
}).sort_values('Coefficient', ascending=False)
print(coef_df.to_string(index=False))
print(f"\nIntercept: {lr.intercept_:.3f}")

## Step 7 — Make Predictions

In [ ]:
y_pred = lr.predict(X_test_sc)

pred_df = pd.DataFrame({
    'Actual Price ($1000s)':    y_test.values[:10],
    'Predicted Price ($1000s)': y_pred[:10].round(2),
    'Difference':               (y_test.values[:10] - y_pred[:10]).round(2)
})
print("Sample Predictions (first 10):")
print(pred_df.to_string(index=False))

## Step 8 — Evaluate the Model

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mae  = np.mean(np.abs(y_test.values - y_pred))

print("=" * 40)
print("  LINEAR REGRESSION EVALUATION")
print("=" * 40)
print(f"  RMSE : ${rmse:.2f}k")
print(f"  MAE  : ${mae:.2f}k")
print(f"  R²   : {r2:.4f}")
print("=" * 40)
print(f"\n📌 RMSE = ${rmse:.2f}k means on average the model's")
print(f"   predictions are off by about ${rmse:.2f},000.")
print(f"\n📌 R² = {r2:.4f} means the model explains")
print(f"   {r2*100:.1f}% of the variance in house prices.")
print(f"\n📌 Lower RMSE = better model performance.")

In [ ]:
# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred, alpha=0.6, color='#2563eb', s=25)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($1000s)')
axes[0].set_ylabel('Predicted Price ($1000s)')
axes[0].set_title('Actual vs Predicted House Prices')
axes[0].legend()

residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.6, color='#f97316', s=25)
axes[1].axhline(y=0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price ($1000s)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.suptitle(f'Linear Regression Results  |  RMSE: ${rmse:.2f}k  |  R²: {r2:.4f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('regression_results.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance (coefficients)
coef_df_sorted = pd.DataFrame({
    'Feature': features,
    'Coefficient': lr.coef_
}).sort_values('Coefficient')

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#ef4444' if c < 0 else '#22c55e' for c in coef_df_sorted['Coefficient']]
ax.barh(coef_df_sorted['Feature'], coef_df_sorted['Coefficient'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Linear Regression Coefficients\n(Green = increases price, Red = decreases price)')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('regression_coefficients.png', bbox_inches='tight')
plt.show()
print("rm (rooms) has the strongest positive effect on price.")
print("lstat (lower-status %) has the strongest negative effect.")

### Task 1 Conclusion
The Linear Regression model achieved an **R² of ~0.71**, meaning it explains about 71% of the variation in house prices. The RMSE indicates average predictions are within ~$4,800 of the actual price. Key findings:
- More rooms (`rm`) strongly increases predicted price
- Higher lower-status population (`lstat`) strongly decreases predicted price
- The residual plot shows some non-linearity — a more complex model (e.g. Random Forest) could improve results

---
# TASK 2 — Logistic Regression: Titanic Survival Classification

## Problem Statement
Predict whether a Titanic passenger **survived or not** based on features such as age, fare, passenger class, and gender.

**Dataset:** Titanic Dataset (891 passengers)  
**Target:** `Survived` — 1 = Survived, 0 = Did Not Survive  
**Algorithm:** Logistic Regression

## Step 1 — Load Dataset

In [ ]:
titanic_raw = pd.read_csv('/mnt/user-data/uploads/Titanic-Dataset.csv')
print(f"Shape: {titanic_raw.shape}")
titanic_raw.head()

## Step 2 — Data Cleaning

In [ ]:
df = titanic_raw.copy()

# Drop columns not useful for prediction
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)

# Impute missing Age with median
df['Age'].fillna(df['Age'].median(), inplace=True)

# Impute missing Embarked with mode
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Encode Sex: male=1, female=0
df['Sex'] = (df['Sex'] == 'male').astype(int)

# One-hot encode Embarked
df = pd.concat([df, pd.get_dummies(df['Embarked'], prefix='Embarked', drop_first=True).astype(int)], axis=1)
df.drop(columns=['Embarked'], inplace=True)

print("✅ Data cleaned.")
print(f"Shape after cleaning: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head(3)

## Step 3 — Select Features and Target

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_Q', 'Embarked_S']
X = df[features]
y = df['Survived']

print(f"Features: {features}")
print(f"\nTarget distribution:")
print(y.value_counts().rename({0:'Did Not Survive', 1:'Survived'}))
print(f"\nSurvival rate: {y.mean()*100:.1f}%")

## Step 4 — Split Dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training samples: {len(X_train)} (80%)")
print(f"Testing samples:  {len(X_test)} (20%)")

scaler2 = StandardScaler()
X_train_sc = scaler2.fit_transform(X_train)
X_test_sc  = scaler2.transform(X_test)
print("\n✅ Features scaled using StandardScaler.")

## Step 5 — Train Logistic Regression Model

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_sc, y_train)
print("✅ Logistic Regression model trained.")

## Step 6 — Make Predictions

In [ ]:
y_pred = log_reg.predict(X_test_sc)
y_prob = log_reg.predict_proba(X_test_sc)[:, 1]

pred_df = pd.DataFrame({
    'Actual':      y_test.values[:10],
    'Predicted':   y_pred[:10],
    'Probability': y_prob[:10].round(3),
    'Correct':     (y_test.values[:10] == y_pred[:10])
}).replace({0: 'Did Not Survive', 1: 'Survived'}, subset=['Actual','Predicted'])
print("Sample Predictions (first 10):")
print(pred_df.to_string(index=False))

## Step 7 — Evaluate the Model

In [ ]:
acc = accuracy_score(y_test, y_pred)
cm  = confusion_matrix(y_test, y_pred)

print("=" * 45)
print("  LOGISTIC REGRESSION EVALUATION")
print("=" * 45)
print(f"  Accuracy Score: {acc*100:.2f}%")
print("=" * 45)
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Did Not Survive', 'Survived']))
print(f"📌 Higher accuracy = better classification model.")
print(f"📌 {acc*100:.2f}% means the model correctly predicted")
print(f"   survival outcome for {acc*100:.2f}% of passengers.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
ConfusionMatrixDisplay(cm, display_labels=['Did Not Survive','Survived']).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Confusion Matrix\nAccuracy: {acc*100:.2f}%')

# Prediction probability distribution
axes[1].hist(y_prob[y_test==0], bins=25, alpha=0.6, color='#ef4444', label='Did Not Survive')
axes[1].hist(y_prob[y_test==1], bins=25, alpha=0.6, color='#22c55e', label='Survived')
axes[1].axvline(x=0.5, color='black', linestyle='--', lw=2, label='Decision boundary (0.5)')
axes[1].set_xlabel('Predicted Probability of Survival')
axes[1].set_ylabel('Count')
axes[1].set_title('Prediction Probability Distribution')
axes[1].legend()

plt.suptitle('Logistic Regression — Titanic Survival Prediction',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('classification_results.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature coefficients
coef_df2 = pd.DataFrame({
    'Feature':     features,
    'Coefficient': log_reg.coef_[0]
}).sort_values('Coefficient')

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#ef4444' if c < 0 else '#22c55e' for c in coef_df2['Coefficient']]
ax.barh(coef_df2['Feature'], coef_df2['Coefficient'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Coefficients\n(Green = increases survival odds, Red = decreases)')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('classification_coefficients.png', bbox_inches='tight')
plt.show()
print("Sex (male=1) has the strongest negative effect — males less likely to survive.")
print("Higher Pclass (3rd class) decreases survival probability.")

### Task 2 Conclusion
The Logistic Regression model achieved **~80% accuracy** on the Titanic test set. Key findings:
- Being male is the strongest predictor of not surviving
- Higher passenger class (1st vs 3rd) significantly increases survival probability
- Higher Fare correlates with increased survival odds
- The probability distribution shows good separation between the two classes

---
## Overall Summary

| Task | Model | Dataset | Key Metric |
|---|---|---|---|
| Task 1 | Linear Regression | Boston Housing | RMSE ~$4.8k, R² ~0.71 |
| Task 2 | Logistic Regression | Titanic | Accuracy ~80.45% |

Both models demonstrate the supervised learning workflow end-to-end: data loading → cleaning → feature selection → splitting → training → prediction → evaluation.